# 🧠 Research Paper Intelligence Engine — RAG System
**Phase 1 of the Agentic AI Research Scientist System**

This notebook allows you to:
1. Upload research paper PDFs
2. Extract and chunk text
3. Generate embeddings with Sentence Transformers
4. Build a FAISS vector index
5. Answer questions via RAG
6. Summarize papers
7. Extract key findings, limitations, and future work
8. Launch the Streamlit UI via ngrok tunnel


## Step 1: Install Dependencies

In [ ]:
%%capture
!pip install PyMuPDF sentence-transformers faiss-cpu langchain langchain-community transformers torch streamlit tqdm numpy pyngrok

## Step 2: Clone or Upload Project Files

Option A — Mount Google Drive (recommended for persistence):

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Set your project folder path in Drive
PROJECT_DIR = '/content/drive/MyDrive/VT'
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f'Project dir: {PROJECT_DIR}')

Option B — Upload from local machine:

In [ ]:
# Upload your project ZIP from local machine
from google.colab import files

# Uncomment to upload a zip of your project:
# uploaded = files.upload()
# !unzip your_project.zip -d /content/VT

PROJECT_DIR = '/content/VT'
print(f'Project dir: {PROJECT_DIR}')

## Step 3: Create Project Structure

In [ ]:
import os

# Update this to your actual project dir
PROJECT_DIR = '/content/VT'

dirs = [
    'data', 'documents', 'embeddings', 'vector_store', 'src'
]
for d in dirs:
    os.makedirs(os.path.join(PROJECT_DIR, d), exist_ok=True)

print('✅ Directories created')
!ls -la {PROJECT_DIR}

## Step 4: Upload Research Papers

In [ ]:
from google.colab import files
import shutil, os

PROJECT_DIR = '/content/VT'
DATA_DIR    = os.path.join(PROJECT_DIR, 'data')

print('Upload your research paper PDF(s):')
uploaded = files.upload()

for fname, data in uploaded.items():
    dest = os.path.join(DATA_DIR, fname)
    with open(dest, 'wb') as f:
        f.write(data)
    print(f'  ✅ Saved: {dest}')

print(f'\nFiles in data/: {os.listdir(DATA_DIR)}')

## Step 5: Test Phase 1 — PDF Extraction

In [ ]:
import sys
sys.path.insert(0, PROJECT_DIR)

from src.pdf_processor import PDFProcessor

processor = PDFProcessor(
    data_dir=os.path.join(PROJECT_DIR, 'data'),
    docs_dir=os.path.join(PROJECT_DIR, 'documents'),
)
results = processor.process_all()

for fname, data in results.items():
    print(f"\n📄 {fname}")
    print(f"   Characters: {len(data['clean_text']):,}")
    print(f"   Saved to  : {data['txt_path']}")
    print(f"   Preview   : {data['clean_text'][:200]}...")

## Step 6: Test Phase 2 — Chunking

In [ ]:
from src.chunking import TextChunker

chunker    = TextChunker(chunk_size=500, chunk_overlap=100)
all_chunks = chunker.chunk_documents(results)

print(f'✅ Total chunks: {len(all_chunks)}')
print(f'\nSample chunk:')
print(all_chunks[0])

## Step 7: Test Phase 3 — Embeddings

In [ ]:
from src.embeddings import EmbeddingGenerator

embed_gen  = EmbeddingGenerator()
embeddings = embed_gen.embed_chunks(all_chunks)
embed_gen.save_embeddings(embeddings)

print(f'✅ Embeddings shape: {embeddings.shape}')

## Step 8: Test Phase 4 — FAISS Index

In [ ]:
from src.vector_db import VectorDB

vdb = VectorDB()
vdb.build_index(embeddings, all_chunks)
vdb.save()

print(f'✅ Index built: {vdb.get_stats()}')

## Step 9: Test Phase 5 — Semantic Retrieval

In [ ]:
from src.retriever import Retriever

retriever = Retriever(embedding_gen=embed_gen, vector_db=vdb)
query     = 'What is the main contribution of this paper?'
results_r = retriever.retrieve(query, top_k=3)

print(f'🔍 Query: {query}')
for r in results_r:
    print(f"\n  Rank {r['rank']} | Score {r['score']:.4f} | {r['source']}")
    print(f"  {r['text'][:150]}...")

## Step 10: Test Phase 6 — RAG Q&A

In [ ]:
from src.rag_pipeline import RAGPipeline

rag    = RAGPipeline(retriever=retriever)
result = rag.answer('What dataset was used for evaluation?')

print(f"\n❓ Question  : {result['question']}")
print(f"✅ Answer    : {result['answer']}")
print(f"📊 Confidence: {result['confidence']:.2%}")
print(f"📄 Sources   : {[s['source'] for s in result['sources']]}")

## Step 11: Test Phase 7 & 8 — Summarization + Insights

In [ ]:
from src.summarizer import Summarizer

summ   = Summarizer()
source = list(results.keys())[0]

analysis = summ.full_analysis(all_chunks, source=source)

print(f"\n📝 SUMMARY\n{'-'*60}")
print(analysis['summary'])

print(f"\n🟢 KEY FINDINGS\n{'-'*60}")
findings = analysis['key_findings']
if isinstance(findings, list):
    for f in findings:
        print(f'  • {f}')
else:
    print(findings[:500])

print(f"\n🟡 LIMITATIONS\n{'-'*60}")
limits = analysis['limitations']
if isinstance(limits, list):
    for l in limits:
        print(f'  • {l}')
else:
    print(limits[:500])

print(f"\n🔵 FUTURE WORK\n{'-'*60}")
future = analysis['future_work']
if isinstance(future, list):
    for fw in future:
        print(f'  • {fw}')
else:
    print(future[:500])

## Step 12: Launch Streamlit UI via ngrok

In [ ]:
# Install ngrok for exposing Streamlit in Colab
!pip install pyngrok -q
from pyngrok import ngrok
import threading, subprocess, time

# Set your ngrok authtoken (get free at https://dashboard.ngrok.com)
NGROK_TOKEN = 'YOUR_NGROK_TOKEN_HERE'  # ← Replace this
ngrok.set_auth_token(NGROK_TOKEN)

# Start Streamlit in background
def run_streamlit():
    os.chdir(PROJECT_DIR)
    subprocess.run(['streamlit', 'run', 'app.py',
                    '--server.port', '8501',
                    '--server.headless', 'true',
                    '--browser.gatherUsageStats', 'false'])

thread = threading.Thread(target=run_streamlit, daemon=True)
thread.start()
time.sleep(5)  # wait for Streamlit to start

# Create ngrok tunnel
public_url = ngrok.connect(8501)
print(f'\n🚀 Streamlit app running at: {public_url}')
print('   Open the URL above in your browser!')